In [1]:
import pandas as pd
import warnings
from model import LinearRegression, ElasticNet, NN, RandomForest, K_Means_NN, XGBoost, CNN
from utils import LoadData
from rolling_train_test import RollingTrainTest
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
factor = pd.read_csv('../CSV/factor_interaction_0.001_R.csv')
input_dim = factor.shape[1] - 2
print(f"Input dimension: {input_dim}")

Input dimension: 86


In [3]:
label = pd.read_csv('../CSV/label_cleaned.csv')
label.describe()

,trade_date,highest_return,lowest_return,close_return
count,324755.000000,324755.000000,324755.000000,324755.000000
mean,201992.644643,10.626280,-8.325162,0.094703
std,279.485259,9.005831,7.293652,10.425341
min,201501.000000,0.000000,-30.000000,-20.000000
25%,201801.000000,3.420000,-11.960000,-6.920000
50%,202007.000000,8.020000,-6.570000,-0.570000
75%,202210.000000,15.760000,-2.570000,6.610000
max,202412.000000,30.000000,0.000000,20.000000


In [4]:
Data = LoadData(factor, label, batch_size=32, num_workers=0, shuffle=True)
model_list = [
    LinearRegression(input_dim, target=3, model_name="LinearRegression"),
    ElasticNet(input_dim, target=3, alpha=0.5, l1_ratio=1, model_name="Lasso"),
    ElasticNet(input_dim, target=3, alpha=0.8, l1_ratio=0.5, model_name="ElasticNet"),
    RandomForest(n_estimators=100, target=3, max_depth=5, model_name="RandomForest"),
    XGBoost(target=3, n_estimators=100, max_depth=5, learning_rate=0.05, model_name="XGBoost"),
    NN(input_dim, target=3, alpha=0.5, l1_ratio=1, layer=2, model_name="NN"),
    NN(input_dim, target=3, alpha=0.5, l1_ratio=1, layer=3, model_name="NN"),
    NN(input_dim, target=3, alpha=0.5, l1_ratio=1, layer=4, model_name="NN"),
    NN(input_dim, target=3, alpha=0.5, l1_ratio=1, layer=5, model_name="NN"),
    K_Means_NN(input_dim, target=3, alpha=0.5, l1_ratio=1, n_clusters=10, layer=5, model_name="K_Means_NN"),
    CNN(input_dim, target=3, model_name="CNN")
]

In [ ]:
count = 1
for model in model_list:
    RTT = RollingTrainTest(model, Data, train_size=0.5, test_size=0.1, epochs=10, patience=3, criterion=None, count=count)
    RTT.info(
        predictability_name = '[interaction_0.001_R | mse_loss | batch=32 | shuffle=True | alpha=0.5 | l1=1 | LeakyReLU(0.3) |FL_None]'
        )
    RTT.run()
    RTT.backtest(trade_mode=1)
    count += 1
    print(f"Model {model.model_name} backtest completed...")
    print("=" * 50)

LinearRegression will run 5 iterations.


  0%|          | 0/10 [00:00<?, ?it/s]